# Prueba final BCD5105 - Cuaderno de celdas

**Instrucciones:** corran cada celda en orden con `Ctrl+Enter`. No modifiquen nada excepto donde el enunciado lo indica (los valores marcados con `<-`). Este cuaderno solo usa `numpy` y `scipy`, ya instalados en Colab. Semilla fija `2026`: sus salidas deben coincidir con las impresas en el enunciado.

**Recuerden desactivar las funciones de asistencia con IA de Colab durante la prueba.**

## Celda A1 - La ruta de Liberia a Roatan

In [5]:
import heapq

# Las 29 rutas reales de rutas.csv (incluidas aqui para que la celda sea autocontenida)
rutas = [("BZE","RTB",234),("BZE","SAL",463),("BZE","SAP",236),("GUA","MGA",544),
         ("GUA","PTY",1358),("GUA","RTB",470),("GUA","SAL",203),("GUA","SAP",296),
         ("GUA","SJO",855),("GUA","TGU",361),("LIR","PTY",696),("LIR","SAL",496),
         ("LIR","SJO",161),("MGA","PTY",816),("MGA","SAL",345),("MGA","SJO",321),
         ("PTY","SAL",1161),("PTY","SAP",1168),("PTY","SJO",539),("PTY","TGU",1018),
         ("RTB","SAL",420),("RTB","SAP",178),("RTB","TGU",262),("SAL","SAP",255),
         ("SAL","SJO",652),("SAL","TGU",210),("SAP","SJO",728),("SAP","TGU",172),
         ("SJO","TGU",558)]

ORIGEN, DESTINO = "SJO", "RTB"          # <- para la parte (c) cambien estos valores

vecinos = {}
for u, v, w in rutas:
    vecinos.setdefault(u, []).append((v, w))
    vecinos.setdefault(v, []).append((u, w))

dist = {n: float("inf") for n in vecinos}
dist[ORIGEN] = 0
anterior = {}
pendientes = [(0, ORIGEN)]
cerrados = []
while pendientes:
    d, u = heapq.heappop(pendientes)
    if u in cerrados:
        continue
    cerrados.append(u)
    for v, w in vecinos[u]:
        if d + w < dist[v]:
            dist[v] = d + w
            anterior[v] = u
            heapq.heappush(pendientes, (dist[v], v))

print(f"Distancias minimas desde {ORIGEN} (km):")
for n in sorted(dist, key=dist.get):
    print(f"   {n}: {dist[n]:6.0f}")
camino = [DESTINO]
while camino[-1] != ORIGEN:
    camino.append(anterior[camino[-1]])
print(f"\nRuta mas corta {ORIGEN} -> {DESTINO}: {' - '.join(reversed(camino))}"
      f"  ({dist[DESTINO]:.0f} km, {len(camino)-2} escala(s))")


Distancias minimas desde SJO (km):
   SJO:      0
   LIR:    161
   MGA:    321
   PTY:    539
   TGU:    558
   SAL:    652
   SAP:    728
   RTB:    820
   GUA:    855
   BZE:    964

Ruta mas corta SJO -> RTB: SJO - TGU - RTB  (820 km, 1 escala(s))


## Celda A2 - Repartir la flota entre Panama y Guatemala

In [9]:
from scipy.optimize import linprog

# Reparto semanal de flota entre SJO-PTY (539 km) y SJO-GUA (855 km)
# x = vuelos redondos a Panama, y = vuelos redondos a Guatemala
# Supuestos: ganancia 500 y 600 USD por vuelo; 3 y 4 horas de flota por vuelo
HORAS = 70          # horas de flota disponibles por semana   <- parte (c): cambien a 70
SLOTS = 18          # salidas semanales que asigna el aeropuerto

resultado = linprog(c=[-500, -600],                 # linprog minimiza: se niega la ganancia
                    A_ub=[[3, 4], [1, 1]],
                    b_ub=[HORAS, SLOTS],
                    bounds=[(0, None), (0, None)],
                    method="highs")

x, y = resultado.x
print(f"Plan optimo: {x:.0f} vuelos a Panama y {y:.0f} a Guatemala por semana")
print(f"Ganancia semanal maxima: {-resultado.fun:,.0f} USD")
print(f"Horas de flota usadas: {3*x + 4*y:.0f} de {HORAS}")
print(f"Salidas usadas: {x + y:.0f} de {SLOTS}")
print()
sombra_horas, sombra_slots = -resultado.ineqlin.marginals
print(f"Valor de una hora extra de flota (precio sombra): {sombra_horas:,.0f} USD")
print(f"Valor de un slot extra de salida (precio sombra): {sombra_slots:,.0f} USD")


Plan optimo: 2 vuelos a Panama y 16 a Guatemala por semana
Ganancia semanal maxima: 10,600 USD
Horas de flota usadas: 70 de 70
Salidas usadas: 18 de 18

Valor de una hora extra de flota (precio sombra): 100 USD
Valor de un slot extra de salida (precio sombra): 200 USD


## Celda A3 - La decision de abrir San Jose-Belice

In [7]:
import numpy as np

# Decision: abrir o no la ruta directa SJO-BZE (hoy no existe en rutas.csv)
# Supuestos: utilidad anual (miles de USD) segun la demanda del primer anio
escenarios    = ["alta", "media", "baja"]
probabilidades = [0.40, 0.35, 0.25]     # <- parte (c): cambien a [0.20, 0.30, 0.50]
utilidades     = [180, 40, -120]        # miles de USD

rng = np.random.default_rng(2026)
ANIOS = 10000
indice = rng.choice(3, size=ANIOS, p=probabilidades)
resultado_abrir = np.array(utilidades)[indice]

print(f"Simulacion de {ANIOS} 'primeros anios' de la ruta SJO-BZE:")
print(f"   utilidad promedio si ABRE la ruta: {resultado_abrir.mean():+.1f} mil USD")
print(f"   utilidad si NO abre: +0.0 mil USD (certeza)")
print(f"   porcentaje de anios con perdida: {100*np.mean(resultado_abrir < 0):.1f}%")
print(f"   peor resultado posible: {resultado_abrir.min()} mil USD")
print()
# Con informacion perfecta: un estudio revela la demanda ANTES de decidir,
# de modo que en el escenario de perdida simplemente no se abre
con_estudio = np.where(np.array(utilidades)[indice] > 0,
                       np.array(utilidades)[indice], 0)
print(f"   utilidad promedio decidiendo CON el estudio previo: {con_estudio.mean():+.1f} mil USD")
print(f"   valor de la informacion (diferencia de promedios): "
      f"{con_estudio.mean() - resultado_abrir.mean():.1f} mil USD")


Simulacion de 10000 'primeros anios' de la ruta SJO-BZE:
   utilidad promedio si ABRE la ruta: +55.0 mil USD
   utilidad si NO abre: +0.0 mil USD (certeza)
   porcentaje de anios con perdida: 25.2%
   peor resultado posible: -120 mil USD

   utilidad promedio decidiendo CON el estudio previo: +85.2 mil USD
   valor de la informacion (diferencia de promedios): 30.3 mil USD


## Celda A4 - Cual promocion mostrar

In [8]:
import numpy as np

# Registro historico (300 visitas): A 58/200 (0.290), B 18/80 (0.225), C 5/20 (0.250).
# Un analista propone quedarse con A para siempre. Antes de decidir, se simula.
# Tasas VERDADERAS de compra (que nadie conoce):
tasas = np.array([0.25, 0.18, 0.40])          # A, B, C  -> la mejor es C
nombres = ["A: 2x1 equipaje", "B: madrugador", "C: millas dobles"]
VISITANTES = 5000                              # <- parte (c): cambien a 50000

def campana(estilo, rng):
    exitos = np.zeros(3); mostradas = np.zeros(3)
    a, b = np.ones(3), np.ones(3)
    for _ in range(VISITANTES):
        if estilo == "terco":          # el que va ganando + 10% al azar, siempre
            medias = np.where(mostradas > 0, exitos/np.maximum(mostradas, 1), 1.0)
            k = int(rng.integers(3)) if rng.random() < 0.1 else int(np.argmax(medias))
        else:                          # el apostador: apuesta segun su corazonada
            k = int(np.argmax(rng.beta(a, b)))
        compra = rng.random() < tasas[k]
        mostradas[k] += 1; exitos[k] += compra
        a[k] += compra; b[k] += (not compra)
    return exitos.sum(), mostradas

rng = np.random.default_rng(2026)
for estilo, etiqueta in [("terco", "EL TERCO"), ("apostador", "EL APOSTADOR")]:
    total = np.zeros(1); reparto = np.zeros(3)
    for _ in range(50):                          # 50 repeticiones para promediar
        c, m = campana(estilo, rng)
        total += c; reparto += m
    total /= 50; reparto /= 50
    print(f"{etiqueta}: {total[0]:.0f} compras en {VISITANTES} visitantes")
    print(f"   % de visitantes que vio la MEJOR promocion (C): "
          f"{100*reparto[2]/VISITANTES:.1f}%")
    print(f"   reparto: " + ", ".join(f"{n.split(':')[0]} {100*m/VISITANTES:.0f}%"
                                       for n, m in zip(nombres, reparto)))
    print()


EL TERCO: 1939 compras en 5000 visitantes
   % de visitantes que vio la MEJOR promocion (C): 91.9%
   reparto: A 4%, B 4%, C 92%

EL APOSTADOR: 1978 compras en 5000 visitantes
   % de visitantes que vio la MEJOR promocion (C): 97.6%
   reparto: A 1%, B 1%, C 98%

